<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Data Interoperability with Unity Catalog</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">3. Centralized Data Processing with External Analytics</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</div>

# 3.1 Lecture Centralized Data Processing with External Analytics

This lesson walks through the first major interoperability use case: using Databricks as the centralized processing hub for data while letting external analytics engines like Snowflake and Trino read the same UC-governed tables through credential vending and the Iceberg REST Catalog.

## Learning Objectives

By the end of this lesson, you will be able to:
- Describe the centralized data processing pattern using Databricks as the hub
- Explain the credential vending flow for secure external access
- Identify the implementation steps for enabling external analytics
- Understand the benefits of centralized processing with decentralized analytics

## A. Context: Centralized Data Processing

In this pattern, Databricks ingests and transforms data in one place while UC governs that data for both Databricks-native and external consumers.

<div style="font-size: 1em; display: flex; align-items: center; justify-content: center; gap: 8px; flex-wrap: wrap; overflow-x: auto; max-width: 100%; margin: 16px 0;">
  <div style="background: #f5f5f5; border: 2px solid #9e9e9e; padding: 14px 18px; border-radius: 6px; text-align: center;">
    <strong>Data Sources</strong><br/><span style="font-size: 0.85em; color: #555;">CDC, Streaming, Batch</span>
  </div>
  <div style="color: #999; font-size: 1.3em;">&#10132;</div>
  <div style="background: #FF3621; border: 2px solid #CC2B1A; color: #fff; padding: 14px 18px; border-radius: 6px; text-align: center;">
    <strong>Databricks</strong><br/>Central Processing Hub<br/><span style="font-size: 0.85em; font-style: italic;">Lakeflow Jobs, Spark Declarative Pipelines</span>
  </div>
  <div style="color: #999; font-size: 1.3em;">&#10132;</div>
  <div style="background: #f3e5f5; border: 2px solid #9c27b0; padding: 14px 18px; border-radius: 6px; text-align: center;">
    <strong>Unity Catalog</strong><br/><span style="font-size: 0.85em; color: #555;">Governance</span>
  </div>
  <div style="color: #999; font-size: 1.3em;">&#8646;</div>
  <div style="display: flex; flex-direction: column; gap: 8px;">
    <div style="background: #e8f5e9; border: 2px solid #4caf50; padding: 10px 14px; border-radius: 6px; text-align: center; width: 220px;">
      <strong>External Analytics</strong><br/><span style="font-size: 0.85em; color: #555;">Snowflake, Trino, ...</span><br/><span style="font-size: 0.75em; color: #999; font-weight: bold;">(via Iceberg REST)</span>
    </div>
    <div style="background: #e3f2fd; border: 2px solid #1976d2; padding: 10px 14px; border-radius: 6px; text-align: center; width: 220px;">
      <strong>Databricks Analytics</strong><br/><span style="font-size: 0.85em; color: #555;">SQL, Notebooks, Dashboards, ...</span>
    </div>
  </div>
</div>

- Use Databricks as the **central data processing hub** with UC as the source of truth
- Allow customers to leverage **external analytics engines** for downstream processing
- Maintain **centralized governance** while enabling external access

## B. Benefits of Centralized Data Processing

| Benefit | Detail |
|---------|--------|
| **Unified Governance** | Single source of truth for all data assets; centralized access control, auditing, and lineage |
| **Enhanced Performance** | Leverage Databricks optimization features (Predictive Optimization, Liquid Clustering) |
| **Operational Flexibility** | Process data where it makes the most sense; enable specialized tools for specific workloads |
| **Simplified Architecture** | Reduce complexity by centralizing metadata; eliminate the need for multiple catalogs |

## C. Implementation Detail

<p style="font-size: 1em; line-height: 1.6; color: #333">The data flow combines ingestion, transformation, and external consumption into a single sequence governed end-to-end by Unity Catalog. The pull-down below shows the order of operations from source ingestion through to external clients reading via the Iceberg REST endpoint.</p>


<br/>
<details>
  <summary style="cursor: pointer; list-style: none; user-select: none">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span>&#x25B6;</span>
        <strong style="font-size: 1.1em; color: #1B5162">Show Sequence Diagram</strong>
      </div>
    </div>
  </summary>

  <div class="mermaid" id="diagram-3-1-credential-vending-seq" style="display: none; font-size: 1em;">
sequenceDiagram
    actor U as User
    participant EX as External System
    participant CV as Credential Vending
    participant UC as Unity Catalog
    U->>+EX: 1. Query
    EX->>+CV: 2. Request Access
    CV->>+UC: 3. Validate Permissions
    UC-->>-CV: Validated
    CV-->>-EX: 4. Issue Tokens
    EX->>+UC: 5. Query via REST
    UC-->>-EX: Data
    EX-->>-U: Results
  </div>

</details>

<script type="module">
import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs";
mermaid.initialize({ startOnLoad: false });

const details = document.querySelector("details");
const node = document.getElementById("diagram-3-1-credential-vending-seq");
if (details && node) {
  let rendered = false;
  details.addEventListener("toggle", async () => {
    if (!details.open || rendered) return;
    rendered = true;
    node.style.display = "block";
    try {
      await mermaid.run({ querySelector: "#diagram-3-1-credential-vending-seq" });
    } catch(e) {
      await new Promise(r => setTimeout(r, 1000));
      await mermaid.run({ querySelector: "#diagram-3-1-credential-vending-seq" });
    }
    node.querySelectorAll('svg text, svg .nodeLabel, svg foreignObject div, svg span').forEach(el => { el.style.fontSize = '1em'; });
  });
}
</script>

## D. Credential Vending Flow

Before an external client can read a UC table through the Iceberg REST endpoint, it has to trade its identity for a short-lived, path-scoped storage credential issued by Unity Catalog. UC also writes an audit record at the moment of authorization - so who-touched-what is captured at the control plane, not at the data plane.

The full handshake is detailed the 2.3 lecture on __External Access to Managed Tables__. Refer to that diagram for the canonical flow.

## E. Reference Implementation (Reading UC Data from Snowflake)

<p style="font-size: 1em; line-height: 1.6; color: #333">Setting up centralized processing with external analytics breaks down into a Databricks-side configuration and a corresponding setup on the external engine.</p>

<div style="font-size: 1em; border-left: 4px solid #607d8b; background: #eceff1; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
    <div style="display: flex; align-items: flex-start; gap: 12px">
        <div>
            <strong style="color: #37474f; font-size: 1.1em">Prerequisites</strong>
            <p style="margin: 8px 0 0 0; color: #333">A user or service principal with the Account Admin role enabled, and an equivalently privileged role in the external system.</p>
        </div>
    </div>
</div>

<div style="font-size: 1em; display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin: 16px 0">

  <div style="background: #FF362133; border: 2px solid #FF3621; border-radius: 8px; padding: 14px 16px">
    <div style="font-weight: 700; color: #B5260E; text-transform: uppercase; margin-bottom: 4px">Databricks &middot; Step 1</div>
    <div style="font-weight: 600; color: #263238; margin-bottom: 8px">Enable External Data Access</div>
    <div style="color: #455a64">Turn on external Iceberg REST access at the metastore level.</div>
  </div>

  <div style="background: #FF362133; border: 2px solid #FF3621; border-radius: 8px; padding: 14px 16px">
    <div style="font-weight: 700; color: #B5260E; text-transform: uppercase; margin-bottom: 4px">Databricks &middot; Step 2</div>
    <div style="font-weight: 600; color: #263238; margin-bottom: 8px">Create Managed Tables</div>
    <div style="color: #455a64">Materialize as managed Iceberg, or as Delta with UniForm.</div>
  </div>

  <div style="background: #FF362133; border: 2px solid #FF3621; border-radius: 8px; padding: 14px 16px">
    <div style="font-weight: 700; color: #B5260E; text-transform: uppercase; margin-bottom: 4px">Databricks &middot; Step 3</div>
    <div style="font-weight: 600; color: #263238; margin-bottom: 8px">Create a Service Principal</div>
    <div style="color: #455a64">Provision the SP and a client secret that the external engine authenticates as.</div>
  </div>

  <div style="background: #FF362133; border: 2px solid #FF3621; border-radius: 8px; padding: 14px 16px">
    <div style="font-weight: 700; color: #B5260E; text-transform: uppercase; margin-bottom: 4px">Databricks &middot; Step 4</div>
    <div style="font-weight: 600; color: #263238; margin-bottom: 8px">Grant Privileges</div>
    <div style="color: #455a64">Grant <code>EXTERNAL USE SCHEMA</code>, <code>SELECT</code>, and <code>USE</code> on the catalog and schema to the SP.</div>
  </div>

  <div style="background: #29B5E833; border: 2px solid #29B5E8; border-radius: 8px; padding: 14px 16px">
    <div style="font-weight: 700; color: #1577AB; text-transform: uppercase; margin-bottom: 4px">Snowflake &middot; Step 1</div>
    <div style="font-weight: 600; color: #263238; margin-bottom: 8px">Create Catalog Integration</div>
    <div style="color: #455a64"><code>CREATE CATALOG INTEGRATION</code> using the SP credentials and the UC Iceberg REST endpoint URL.</div>
  </div>

  <div style="background: #29B5E833; border: 2px solid #29B5E8; border-radius: 8px; padding: 14px 16px">
    <div style="font-weight: 700; color: #1577AB; text-transform: uppercase; margin-bottom: 4px">Snowflake &middot; Step 2</div>
    <div style="font-weight: 600; color: #263238; margin-bottom: 8px">Create Iceberg Tables</div>
    <div style="color: #455a64"><code>CREATE ICEBERG TABLE</code> in Snowflake referencing the catalog integration.</div>
  </div>

  <div style="background: #29B5E833; border: 2px solid #29B5E8; border-radius: 8px; padding: 14px 16px">
    <div style="font-weight: 700; color: #1577AB; text-transform: uppercase; margin-bottom: 4px">Snowflake &middot; Step 3</div>
    <div style="font-weight: 600; color: #263238; margin-bottom: 8px">Query UC Tables Natively</div>
    <div style="color: #455a64"><code>SELECT</code> from the Iceberg tables exactly like any other Snowflake table.</div>
  </div>

  <div></div>

</div>

<div style="font-size: 1em; border-left: 4px solid #009688; background: #e0f2f1; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
    <div style="display: flex; align-items: flex-start; gap: 12px">
        <div>
            <strong style="color: #00695c; font-size: 1.1em">Other Systems</strong>
            <p style="margin: 8px 0 0 0; color: #333">The same pattern applies to Trino, Dremio, Starburst, and other Iceberg-compatible engines. Each system has its own catalog integration syntax.</p>
        </div>
    </div>
</div>

## Key Takeaways

- **Centralized processing** keeps data governance unified while enabling flexible analytics
- **Credential vending** provides secure, time-limited access to external systems
- The **Iceberg REST Catalog** is the key integration point for external analytics engines
- Implementation requires **service principals** with appropriate UC privileges

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>